<a href="https://colab.research.google.com/github/Indu7777/6thSem-BDA-Lab/blob/main/hospital_emergency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# AVL Tree Node (Patient)
class Node:
    def __init__(self, name, severity):
        self.name = name
        self.severity = severity
        self.left = None
        self.right = None
        self.height = 1

class AVLTree:

    def height(self, node):
        return node.height if node else 0

    def balance(self, node):
        return self.height(node.left) - self.height(node.right) if node else 0

    def right_rotate(self, y):
        print(f"🔁 Right Rotation on {y.name} (LL case)")
        x = y.left
        T2 = x.right

        x.right = y
        y.left = T2

        y.height = 1 + max(self.height(y.left), self.height(y.right))
        x.height = 1 + max(self.height(x.left), self.height(x.right))

        return x

    def left_rotate(self, x):
        print(f"🔁 Left Rotation on {x.name} (RR case)")
        y = x.right
        T2 = y.left

        y.left = x
        x.right = T2

        x.height = 1 + max(self.height(x.left), self.height(x.right))
        y.height = 1 + max(self.height(y.left), self.height(y.right))

        return y

    # Insert based on severity (higher severity → right side)
    def insert(self, root, name, severity):
        if not root:
            print(f"Inserted Patient {name} (Severity {severity})")
            return Node(name, severity)

        if severity < root.severity:
            root.left = self.insert(root.left, name, severity)
        else:
            root.right = self.insert(root.right, name, severity)

        root.height = 1 + max(self.height(root.left), self.height(root.right))
        balance = self.balance(root)

        # LL
        if balance > 1 and severity < root.left.severity:
            return self.right_rotate(root)

        # RR
        if balance < -1 and severity > root.right.severity:
            return self.left_rotate(root)

        # LR
        if balance > 1 and severity > root.left.severity:
            print("🔁 LR Rotation")
            root.left = self.left_rotate(root.left)
            return self.right_rotate(root)

        # RL
        if balance < -1 and severity < root.right.severity:
            print("🔁 RL Rotation")
            root.right = self.right_rotate(root.right)
            return self.left_rotate(root)

        return root

    # Get highest severity patient (rightmost node)
    def get_max(self, root):
        current = root
        while current.right:
            current = current.right
        return current

    # Delete node
    def delete(self, root, severity):
        if not root:
            return root

        if severity < root.severity:
            root.left = self.delete(root.left, severity)
        elif severity > root.severity:
            root.right = self.delete(root.right, severity)
        else:
            if not root.left:
                return root.right
            elif not root.right:
                return root.left

            temp = self.get_max(root.left)
            root.severity = temp.severity
            root.name = temp.name
            root.left = self.delete(root.left, temp.severity)

        root.height = 1 + max(self.height(root.left), self.height(root.right))
        balance = self.balance(root)

        # Rebalance
        if balance > 1 and self.balance(root.left) >= 0:
            return self.right_rotate(root)

        if balance < -1 and self.balance(root.right) <= 0:
            return self.left_rotate(root)

        if balance > 1 and self.balance(root.left) < 0:
            root.left = self.left_rotate(root.left)
            return self.right_rotate(root)

        if balance < -1 and self.balance(root.right) > 0:
            root.right = self.right_rotate(root.right)
            return self.left_rotate(root)

        return root

    # Inorder display
    def inorder(self, root):
        if root:
            self.inorder(root.left)
            print(f"{root.name}(S={root.severity}, H={root.height}, BF={self.balance(root)})")
            self.inorder(root.right)


# ---------------- DEMO ----------------

tree = AVLTree()
root = None

print("=== PATIENT ARRIVAL ===")
patients = [
    ("A", 2),
    ("B", 5),
    ("C", 3),
    ("D", 8),
    ("E", 1)
]

for p in patients:
    root = tree.insert(root, *p)

print("\n=== AVL TREE (INORDER) ===")
tree.inorder(root)

print("\n=== TREATMENT ORDER ===")
while root:
    max_patient = tree.get_max(root)
    print(f"Treating Patient {max_patient.name} (Severity {max_patient.severity})")
    root = tree.delete(root, max_patient.severity)

    print("Updated Tree:")
    tree.inorder(root)
    print("------")

# Emergency case
print("\n=== EMERGENCY CASE ===")
root = None
root = tree.insert(root, "X", 3)
root = tree.insert(root, "Y", 4)

print("Emergency patient arrives!")
root = tree.insert(root, "Z (CRITICAL)", 10)

tree.inorder(root)

max_patient = tree.get_max(root)
print(f"\nNext treated: {max_patient.name}")

=== PATIENT ARRIVAL ===
Inserted Patient A (Severity 2)
Inserted Patient B (Severity 5)
Inserted Patient C (Severity 3)
🔁 RL Rotation
🔁 Right Rotation on B (LL case)
🔁 Left Rotation on A (RR case)
Inserted Patient D (Severity 8)
Inserted Patient E (Severity 1)

=== AVL TREE (INORDER) ===
E(S=1, H=1, BF=0)
A(S=2, H=2, BF=1)
C(S=3, H=3, BF=0)
B(S=5, H=2, BF=-1)
D(S=8, H=1, BF=0)

=== TREATMENT ORDER ===
Treating Patient D (Severity 8)
Updated Tree:
E(S=1, H=1, BF=0)
A(S=2, H=2, BF=1)
C(S=3, H=3, BF=1)
B(S=5, H=1, BF=0)
------
Treating Patient B (Severity 5)
🔁 Right Rotation on C (LL case)
Updated Tree:
E(S=1, H=1, BF=0)
A(S=2, H=2, BF=0)
C(S=3, H=1, BF=0)
------
Treating Patient C (Severity 3)
Updated Tree:
E(S=1, H=1, BF=0)
A(S=2, H=2, BF=1)
------
Treating Patient A (Severity 2)
Updated Tree:
E(S=1, H=1, BF=0)
------
Treating Patient E (Severity 1)
Updated Tree:
------

=== EMERGENCY CASE ===
Inserted Patient X (Severity 3)
Inserted Patient Y (Severity 4)
Emergency patient arrives!
Ins